# Pandas and NumPy Assignment - Part B (NumPy)

**IT Services Ticket Data**

Notebook 2 of 2. Every cell is numbered with its question number.

Part B works on the numeric matrix taken from the cleaned table that Part A saved:
`hours logged, billing rate, billable amount, resolution hours, CSAT score`, in that order.
It is built once in the setup cell and reused all the way down.

**On Google Colab:** run cell Q0 first. If `it_services_tickets_clean.csv` is not there it
rebuilds it from the raw file automatically, so this notebook runs on its own.


In [1]:
# Q0 - get the cleaned file into Colab
# Part B works on the cleaned table that Part A saved. If it is not here, this
# cell rebuilds it from the raw file using exactly the same cleaning steps,
# so this notebook can be run on its own.
import os
import numpy as np
import pandas as pd

CLEAN = "it_services_tickets_clean.csv"
RAW = "it_services_tickets_raw.csv"


def clean_priority(value):
    if pd.isna(value):
        return np.nan
    t = str(value).strip().upper()
    if "P1" in t or t in ("1", "CRITICAL"):
        return "P1"
    if "P2" in t or t in ("2", "HIGH"):
        return "P2"
    if "P3" in t or t in ("3", "MEDIUM"):
        return "P3"
    if "P4" in t or t in ("4", "LOW"):
        return "P4"
    return np.nan


def rebuild_clean(raw_path):
    """The whole of Part A's cleaning, condensed into one function."""
    df = pd.read_csv(raw_path)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

    tokens = ["NA", "N/A", "null", "-", "NULL", "na", "n/a", "None", "none",
              "nan", "NaN", "#N/A", "missing", "?", "--", ""]
    text_cols = df.select_dtypes(include="object").columns
    df[text_cols] = df[text_cols].apply(
        lambda c: c.str.replace("\t", " ", regex=False).str.strip())
    df = df[df["record_id"] != "Record_ID"]
    df = df.replace(tokens, np.nan).drop_duplicates().reset_index(drop=True)

    status_map = {
        "resolved": "Resolved", "closed": "Closed", "auto-closed": "Closed",
        "open": "Open", "in progress": "In Progress", "in-progress": "In Progress",
        "wip": "In Progress", "on hold": "On Hold", "on-hold": "On Hold",
        "pending customer": "On Hold", "cancelled": "Cancelled",
        "canceled": "Cancelled",
    }
    df["status"] = df["status"].str.strip().str.lower().map(status_map)
    df["priority"] = df["priority"].map(clean_priority)
    df["serviceline"] = df["serviceline"].str.strip().str.title()

    country_fix = {"Us": "USA", "Usa": "USA", "U.S.A.": "USA", "United States": "USA",
                   "Uk": "UK", "U.K.": "UK", "United Kingdom": "UK", "Great Britain": "UK"}
    df["client_country"] = df["client_country"].str.strip().str.title().replace(country_fix)

    def money(col):
        return pd.to_numeric(col.str.replace("$", "", regex=False)
                                .str.replace("USD", "", regex=False)
                                .str.replace(",", "", regex=False)
                                .str.strip(), errors="coerce")

    df["hours_logged"] = pd.to_numeric(
        df["hours_logged"].str.replace("hrs", "", regex=False)
                          .str.replace("hr", "", regex=False).str.strip(),
        errors="coerce")
    df["billing_rate_(usd)"] = money(df["billing_rate_(usd)"])
    df["billable_amount_usd"] = money(df["billable_amount_usd"])
    df["csat_score"] = pd.to_numeric(df["csat_score"], errors="coerce")

    df["client_name"] = (df["client_name"].str.replace("\t", " ", regex=False)
                                          .str.replace(r"\s+", " ", regex=True)
                                          .str.strip().str.title())

    created = pd.to_datetime(df["ticket_created_at"], format="mixed", errors="coerce")
    codes = pd.to_numeric(df["ticket_created_at"], errors="coerce")
    codes = codes.where(codes.between(1_000_000, 2_000_000))
    df["ticket_created_at"] = created.fillna(pd.to_datetime(codes * 1000, unit="s"))

    df["ticket_year"] = df["ticket_created_at"].dt.year
    df["ticket_month"] = df["ticket_created_at"].dt.month
    df["total"] = df["billing_rate_(usd)"] * df["hours_logged"]
    df["csat_score"] = df["csat_score"].fillna(df["csat_score"].mean())
    df = df.dropna(subset=["client_name"])
    df["sla_breached"] = (df["sla_breached"].astype(str).str.strip().str.lower()
                          .isin(["yes", "y", "true", "1"]))
    return df


if not os.path.exists(CLEAN):
    if not os.path.exists(RAW):
        from google.colab import files
        uploaded = files.upload()          # upload it_services_tickets_raw.csv
        RAW = list(uploaded.keys())[0]
    print("Cleaned file not found - rebuilding it from", RAW)
    rebuild_clean(RAW).to_csv(CLEAN, index=False)

print("Using cleaned file:", CLEAN)


Using cleaned file: it_services_tickets_clean.csv


## Setup

In [2]:
# Setup
import numpy as np
import pandas as pd

clean = pd.read_csv(CLEAN)
clean["resolution_time_hours"] = pd.to_numeric(clean["resolution_time_hours"],
                                               errors="coerce")

cols = ["hours_logged", "billing_rate_(usd)", "billable_amount_usd",
        "resolution_time_hours", "csat_score"]
M = clean[cols].to_numpy(dtype=float)

hours  = M[:, 0]
rate   = M[:, 1]
amount = M[:, 2]
restime = M[:, 3]
csat   = M[:, 4]

print("Matrix M shape:", M.shape)
print("Column order:", cols)

Matrix M shape: (48020, 5)
Column order: ['hours_logged', 'billing_rate_(usd)', 'billable_amount_usd', 'resolution_time_hours', 'csat_score']


### Q1. Import NumPy and print its version.

In [3]:
# Q1
import numpy as np
print(np.__version__)

1.26.4


### Q2. Make an array from the hours logged column.

In [4]:
# Q2
hours = clean["hours_logged"].to_numpy(dtype=float)
print(hours[:10])

[  nan  1.34  7.   34.63 12.76 15.3    nan  9.    1.66 11.1 ]


### Q3. Print the shape, the size and the data type of that array.

In [5]:
# Q3
print("Shape:", hours.shape)
print("Size:", hours.size)
print("Dtype:", hours.dtype)

Shape: (48020,)
Size: 48020
Dtype: float64


### Q4. Make an array of the numbers 1 to 50 using arange.

In [6]:
# Q4
a = np.arange(1, 51)
print(a)

[ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50]


### Q5. Make 10 evenly spaced numbers between 0 and 100 using linspace.

In [7]:
# Q5
b = np.linspace(0, 100, 10)
print(b)

[  0.          11.11111111  22.22222222  33.33333333  44.44444444
  55.55555556  66.66666667  77.77777778  88.88888889 100.        ]


### Q6. Make an array of 10 zeros and an array of 10 ones.

In [8]:
# Q6
print(np.zeros(10))
print(np.ones(10))

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


### Q7. Make a 3 by 3 array of random numbers using seed 42.

In [9]:
# Q7
np.random.seed(42)
r = np.random.rand(3, 3)
print(r)

[[0.37454012 0.95071431 0.73199394]
 [0.59865848 0.15601864 0.15599452]
 [0.05808361 0.86617615 0.60111501]]


### Q8. Make a 2-D array from the hours logged and billing rate columns.

In [10]:
# Q8
rate = clean["billing_rate_(usd)"].to_numpy(dtype=float)
two_d = np.column_stack((hours, rate))
print(two_d[:5])
print("Shape:", two_d.shape)

[[  nan 39.45]
 [ 1.34 33.15]
 [ 7.   48.1 ]
 [34.63 52.98]
 [12.76 22.9 ]]
Shape: (48020, 2)


### Q9. Print how many dimensions that 2-D array has.

In [11]:
# Q9
print("Dimensions:", two_d.ndim)

Dimensions: 2


### Q10. Print the first 5 values of the hours array.

In [12]:
# Q10
print(hours[:5])

[  nan  1.34  7.   34.63 12.76]


### Q11. Print the last 5 values of the hours array.

In [13]:
# Q11
print(hours[-5:])

[50.   4.  47.   2.   6.5]


### Q12. Print every second value from the first 20 values.

In [14]:
# Q12
print(hours[:20:2])

[   nan   7.    12.76    nan   1.66    nan  11.63  28.     0.59 253.85]


### Q13. Reverse the array.

In [15]:
# Q13
print(hours[::-1][:10])
print("Length still:", hours[::-1].size)

[ 6.5   2.   47.    4.   50.    7.21 25.23 19.   37.16   nan]
Length still: 48020


### Q14. Find the sum of the hours array.

In [16]:
# Q14
print("Sum with NaN:", hours.sum())
print("Sum ignoring NaN:", np.nansum(hours))

Sum with NaN: nan
Sum ignoring NaN: 2821031.95


### Q15. Find the mean, the minimum and the maximum of the hours array.

In [17]:
# Q15
print("Mean:", np.nanmean(hours))
print("Min:", np.nanmin(hours))
print("Max:", np.nanmax(hours))

Mean: 62.542277080654465
Min: -12.5
Max: 9888.01


### Q16. Find the standard deviation.

In [18]:
# Q16
print("Std:", np.nanstd(hours))

Std: 483.200788180708


### Q17. Find the median.

In [19]:
# Q17
print("Median:", np.nanmedian(hours))

Median: 13.14


### Q18. Count how many values are missing in the array.

In [20]:
# Q18
print("Missing values:", np.isnan(hours).sum())

Missing values: 2914


### Q19. Find the mean while ignoring the missing values.

In [21]:
# Q19
print("Mean ignoring NaN:", np.nanmean(hours))

Mean ignoring NaN: 62.542277080654465


### Q20. Replace all the missing values with 0.

In [22]:
# Q20
hours_clean = np.nan_to_num(hours, nan=0.0)
print("Missing values now:", np.isnan(hours_clean).sum())
print("New mean:", hours_clean.mean())

Missing values now: 0
New mean: 58.747021032902964


### Q21. Count how many values are greater than 100.

In [23]:
# Q21
print("Values over 100:", (hours_clean > 100).sum())

Values over 100: 1991


### Q22. Show only the values that are greater than 100.

In [24]:
# Q22
print(hours_clean[hours_clean > 100])

[253.85 541.3  128.   ... 562.22 141.9  137.56]


### Q23. Show the values that are between 10 and 50.

In [25]:
# Q23
mask = (hours_clean >= 10) & (hours_clean <= 50)
print("Count:", mask.sum())
print(hours_clean[mask][:20])

Count: 21221
[34.63 12.76 15.3  11.1  11.63 28.   10.16 32.71 30.84 18.   31.44 24.46
 39.   13.   30.02 11.51 20.22 24.15 27.   11.26]


### Q24. Count how many values are negative.

In [26]:
# Q24
print("Negative values:", (hours_clean < 0).sum())

Negative values: 184


### Q25. Replace all the negative values with 0.

In [27]:
# Q25
hours_pos = np.where(hours_clean < 0, 0, hours_clean)
print("Negative values now:", (hours_pos < 0).sum())
print("New min:", hours_pos.min())

Negative values now: 0
New min: 0.0


### Q26. Sort the array from smallest to largest.

In [28]:
# Q26
sorted_hours = np.sort(hours_pos)
print("First 10:", sorted_hours[:10])
print("Last 10:", sorted_hours[-10:])

First 10: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
Last 10: [8760.   8760.   8760.   8760.   8760.   8760.   8760.   8760.   9888.01
 9888.01]


### Q27. Find the position of the largest value.

In [29]:
# Q27
print("Index of max:", hours_pos.argmax(), "| value:", hours_pos.max())

Index of max: 10032 | value: 9888.01


### Q28. Find the position of the smallest value.

In [30]:
# Q28
print("Index of min:", hours_pos.argmin(), "| value:", hours_pos.min())

Index of min: 0 | value: 0.0


### Q29. Find the 5 largest values.

In [31]:
# Q29
print(np.sort(hours_pos)[-5:][::-1])

[9888.01 9888.01 8760.   8760.   8760.  ]


### Q30. Round every value to 1 decimal place.

In [32]:
# Q30
print(np.round(hours_pos, 1)[:10])

[ 0.   1.3  7.  34.6 12.8 15.3  0.   9.   1.7 11.1]


### Q31. Multiply every value by 2.

In [33]:
# Q31
print((hours_pos * 2)[:10])

[ 0.    2.68 14.   69.26 25.52 30.6   0.   18.    3.32 22.2 ]


### Q32. Add 10 to every value.

In [34]:
# Q32
print((hours_pos + 10)[:10])

[10.   11.34 17.   44.63 22.76 25.3  10.   19.   11.66 21.1 ]


### Q33. Multiply the hours array by the billing rate array.

In [35]:
# Q33
rate_pos = np.nan_to_num(rate, nan=0.0)
revenue = hours_pos * rate_pos
print(revenue[:10])
print("Total:", revenue.sum())

[   0.       44.421   336.7    1834.6974  292.204  1828.35      0.
  432.54    120.1342  313.353 ]
Total: 307511078.2406


### Q34. Check whether the two arrays have the same length.

In [36]:
# Q34
print("hours:", hours_pos.size, "| rate:", rate_pos.size)
print("Same length:", hours_pos.size == rate_pos.size)

hours: 48020 | rate: 48020
Same length: True


### Q35. Use np.where to mark values above 24 as slow and the rest as fast.

In [37]:
# Q35
labels = np.where(hours_pos > 24, "slow", "fast")
print(labels[:10])
print("slow:", (labels == "slow").sum(), "| fast:", (labels == "fast").sum())

['fast' 'fast' 'fast' 'slow' 'fast' 'fast' 'fast' 'fast' 'fast' 'fast']
slow: 14606 | fast: 33414


### Q36. Make an array of the unique priority values.

In [38]:
# Q36
prio = clean["priority"].astype(str).to_numpy()
print(np.unique(prio))

['P1' 'P2' 'P3' 'P4' 'nan']


### Q37. Count how many times each unique priority value appears.

In [39]:
# Q37
values, counts = np.unique(prio, return_counts=True)
for v, c in zip(values, counts):
    print(v, c)

P1 11687
P2 11668
P3 11760
P4 11425
nan 1480


### Q38. Take the first 100 values and reshape them into a 10 by 10 array.

In [40]:
# Q38
grid = hours_pos[:100].reshape(10, 10)
print(grid)
print("Shape:", grid.shape)

[[0.00000e+00 1.34000e+00 7.00000e+00 3.46300e+01 1.27600e+01 1.53000e+01
  0.00000e+00 9.00000e+00 1.66000e+00 1.11000e+01]
 [0.00000e+00 1.44000e+00 1.16300e+01 7.23100e+01 2.80000e+01 1.01600e+01
  5.90000e-01 7.00000e+00 2.53850e+02 4.20000e+00]
 [5.08100e+01 8.01000e+00 6.79400e+01 0.00000e+00 3.27100e+01 5.41300e+02
  6.91000e+00 3.08400e+01 3.51000e+00 4.34000e+00]
 [1.80000e+01 3.14400e+01 3.42000e+00 2.44600e+01 3.90000e+01 1.30000e+01
  3.00200e+01 1.15100e+01 6.30200e+01 1.03000e+00]
 [2.90000e+00 3.92000e+00 2.02200e+01 2.41500e+01 8.40000e+01 0.00000e+00
  2.70000e+01 5.20000e+00 4.00000e+00 2.76000e+00]
 [1.12600e+01 2.40000e+01 5.26000e+00 1.40000e+01 5.00000e+00 1.14100e+01
  0.00000e+00 1.80000e+01 1.28000e+02 1.54000e+01]
 [1.90066e+03 4.00000e+00 3.00000e+00 9.00000e+00 1.33350e+02 1.29170e+03
  4.00000e+00 0.00000e+00 2.00000e+00 0.00000e+00]
 [2.05100e+01 1.13300e+01 5.99100e+01 1.00000e+00 2.29700e+01 0.00000e+00
  2.49000e+01 3.34000e+01 3.40000e+01 1.49200e+01]


### Q39. Transpose that 10 by 10 array.

In [41]:
# Q39
print(grid.T)

[[0.00000e+00 0.00000e+00 5.08100e+01 1.80000e+01 2.90000e+00 1.12600e+01
  1.90066e+03 2.05100e+01 1.91700e+01 1.30000e+01]
 [1.34000e+00 1.44000e+00 8.01000e+00 3.14400e+01 3.92000e+00 2.40000e+01
  4.00000e+00 1.13300e+01 1.78600e+01 1.09100e+01]
 [7.00000e+00 1.16300e+01 6.79400e+01 3.42000e+00 2.02200e+01 5.26000e+00
  3.00000e+00 5.99100e+01 9.21000e+00 8.00000e+00]
 [3.46300e+01 7.23100e+01 0.00000e+00 2.44600e+01 2.41500e+01 1.40000e+01
  9.00000e+00 1.00000e+00 5.40000e+01 3.81000e+00]
 [1.27600e+01 2.80000e+01 3.27100e+01 3.90000e+01 8.40000e+01 5.00000e+00
  1.33350e+02 2.29700e+01 0.00000e+00 3.73680e+02]
 [1.53000e+01 1.01600e+01 5.41300e+02 1.30000e+01 0.00000e+00 1.14100e+01
  1.29170e+03 0.00000e+00 2.00000e+01 1.13800e+01]
 [0.00000e+00 5.90000e-01 6.91000e+00 3.00200e+01 2.70000e+01 0.00000e+00
  4.00000e+00 2.49000e+01 6.00000e+00 3.36700e+01]
 [9.00000e+00 7.00000e+00 3.08400e+01 1.15100e+01 5.20000e+00 1.80000e+01
  0.00000e+00 3.34000e+01 1.15600e+01 5.80000e-01]


### Q40. Flatten it back into a 1-D array.

In [42]:
# Q40
flat = grid.flatten()
print(flat[:20])
print("Shape:", flat.shape)

[  0.     1.34   7.    34.63  12.76  15.3    0.     9.     1.66  11.1
   0.     1.44  11.63  72.31  28.    10.16   0.59   7.   253.85   4.2 ]
Shape: (100,)


### Q41. Join two arrays together using np.concatenate.

In [43]:
# Q41
joined = np.concatenate([hours_pos[:5], rate_pos[:5]])
print(joined)
print("Length:", joined.size)

[ 0.    1.34  7.   34.63 12.76 39.45 33.15 48.1  52.98 22.9 ]
Length: 10


### Q42. Split an array into 5 equal parts.

In [44]:
# Q42
parts = np.array_split(hours_pos, 5)
for i, p in enumerate(parts, start=1):
    print("Part", i, "| length", p.size, "| first value", p[0])

Part 1 | length 9604 | first value 0.0
Part 2 | length 9604 | first value 0.25
Part 3 | length 9604 | first value 66.3
Part 4 | length 9604 | first value 26.0
Part 5 | length 9604 | first value 8760.0


### Q43. Find the 25th, 50th and 75th percentile.

In [45]:
# Q43
print(np.percentile(hours_pos, [25, 50, 75]))

[ 3.4   11.595 28.3  ]


### Q44. Find the cumulative sum of the first 20 values.

In [46]:
# Q44
print(np.cumsum(hours_pos[:20]))

[  0.     1.34   8.34  42.97  55.73  71.03  71.03  80.03  81.69  92.79
  92.79  94.23 105.86 178.17 206.17 216.33 216.92 223.92 477.77 481.97]


### Q45. Find the difference between each value and the one before it using np.diff.

In [47]:
# Q45
print(np.diff(hours_pos)[:10])
print("Length:", np.diff(hours_pos).size)

[  1.34   5.66  27.63 -21.87   2.54 -15.3    9.    -7.34   9.44 -11.1 ]
Length: 48019


### Q46. Keep every value inside the range 0 to 100 using np.clip.

In [48]:
# Q46
clipped = np.clip(hours_pos, 0, 100)
print("Min:", clipped.min(), "| Max:", clipped.max())
print(clipped[:10])

Min: 0.0 | Max: 100.0
[ 0.    1.34  7.   34.63 12.76 15.3   0.    9.    1.66 11.1 ]


### Q47. Convert the array to whole numbers.

In [49]:
# Q47
ints = hours_pos.astype(int)
print(ints[:10])
print("Dtype:", ints.dtype)

[ 0  1  7 34 12 15  0  9  1 11]
Dtype: int32


### Q48. Copy the array, change one value in the copy, and show that the original did not change.

In [50]:
# Q48
original = hours_pos[:5].copy()
copied = original.copy()
copied[0] = 999
print("Original:", original)
print("Copy:", copied)

Original: [ 0.    1.34  7.   34.63 12.76]
Copy: [999.     1.34   7.    34.63  12.76]


### Q49. Save the array to a .npy file and load it back.

In [51]:
# Q49
np.save("hours_array.npy", hours_pos)
loaded = np.load("hours_array.npy")
print("Loaded shape:", loaded.shape)
print("Identical to original:", np.array_equal(hours_pos, loaded))

Loaded shape: (48020,)
Identical to original: True


### Q50. Find the total revenue using np.dot on the hours array and the rate array.

In [52]:
# Q50
total_revenue = np.dot(hours_pos, rate_pos)
print("Total revenue (USD):", total_revenue)
print("Same as sum of products:", np.isclose(total_revenue, (hours_pos * rate_pos).sum()))

Total revenue (USD): 307511078.2406004
Same as sum of products: True
